# 03 · Un petit workflow de machine learning
Question : la longueur et la largeur des pétales peuvent-elles aider à distinguer **versicolor** de **virginica** ?

Objectifs : définir des features et une cible, séparer les données, comparer un modèle entraîné à une baseline et relier les tables pandas aux calculs NumPy.

Ce notebook recharge ses propres données et peut s’exécuter indépendamment. L’exploration de l’ensemble des données dans le notebook 02 est un exercice pédagogique distinct. Ici, le choix des features est fixé à l’avance, et nous réservons un jeu de validation avant l’entraînement.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss

iris = load_iris(as_frame=True)
df = iris.frame.loc[iris.frame["target"].isin([1, 2])].copy()
features = ["petal length (cm)", "petal width (cm)"]
X = df[features].to_numpy()
y = (df["target"] == 2).astype(int).to_numpy()
print("X:", X.shape, "y:", y.shape)
print("0 = versicolor, 1 = virginica")
assert X.shape == (100, 2) and y.shape == (100,)

## A. Entraînement et validation
Les données d’entraînement servent à ajuster les paramètres. Les données de validation permettent d’évaluer ou de comparer des choix. Pour un rapport final après la sélection du modèle, réservez un jeu de test distinct.

La seed rend cette séparation reproductible. La stratification conserve les proportions des classes. Une séparation aléatoire convient à ce petit exemple ; des données comportant plusieurs observations par personne, des groupes ou un ordre temporel nécessitent une séparation qui respecte cette structure.

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print("Train:", X_train.shape, "Validation:", X_valid.shape)
assert len(X_train) + len(X_valid) == 100

## B. Une baseline et un modèle entraîné
La baseline prédit toujours la classe la plus fréquente dans les données d’entraînement. La régression logistique utilise les valeurs des features pour estimer une probabilité.

Le pipeline apprend les paramètres de mise à l’échelle uniquement à partir des données d’entraînement. Il applique ces mêmes paramètres aux données de validation. Ajuster le scaler avant la séparation provoquerait une fuite d’information.

Nous utilisons scikit-learn pour nous concentrer sur le workflow.

In [ ]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model.fit(X_train, y_train)

baseline_prediction = baseline.predict(X_valid)
prediction = model.predict(X_valid)
probability = model.predict_proba(X_valid)[:, 1]
results = pd.DataFrame({
    "model": ["Majority baseline", "Logistic regression"],
    "validation accuracy": [accuracy_score(y_valid, baseline_prediction),
                            accuracy_score(y_valid, prediction)],
})
print(results.to_string(index=False))
print("Mean validation log loss:", log_loss(y_valid, probability))

## C. Le calcul NumPy derrière une probabilité
`X @ w + b` produit un score pour chaque ligne. La sigmoïde transforme un score en probabilité.

Utilisez les features **mises à l’échelle**, car les poids appris correspondent à ce système de coordonnées. Les scores de la démonstration sont modérés : l’expression exponentielle directe convient donc ici. Des entrées plus extrêmes nécessitent une sigmoïde numériquement stable.

In [ ]:
scaler, classifier = model.steps[0][1], model.steps[1][1]
X_scaled = scaler.transform(X_valid)
w = classifier.coef_[0]
b = classifier.intercept_[0]
scores = X_scaled @ w + b
probability_numpy = 1 / (1 + np.exp(-scores))
print("Shapes:", X_scaled.shape, w.shape, scores.shape)
np.testing.assert_allclose(probability_numpy, probability)
print("NumPy and scikit-learn probabilities agree.")

### Exercice G · Examiner les erreurs (6 min)
1. Construisez une table pandas avec la vraie classe, la prédiction et la probabilité prédite de virginica.
2. Filtrez les lignes dont la prédiction est incorrecte.
3. Expliquez une erreur : la probabilité était-elle proche de 0.5 ou le modèle était-il confiant ?

Vérification : la table contient 25 lignes. Le nombre d’erreurs doit correspondre au calcul de l’accuracy. Un score élevé sur 25 fleurs n’apporte qu’une preuve limitée concernant d’autres populations.

Facultatif : essayez une seule feature, puis comparez les résultats de validation. Utiliser ce jeu de façon répétée pour choisir les features l’intègre au processus de sélection du modèle. Il ne peut alors plus servir de test final resté intact.

In [ ]:
# Build the result table and inspect mistakes here.

## Questions
Que représentent une ligne de `X` et une valeur de `y` ? Pourquoi comparer le modèle à une baseline ? Quelles données ont servi à ajuster le scaler ? Que faudrait-il sauvegarder pour qu’une autre personne puisse reproduire cette exécution ?

## Sources
[Pièges courants et pipelines scikit-learn](https://scikit-learn.org/stable/common_pitfalls.html), [dataset Iris](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_iris.html). Lectures : *An Introduction to Statistical Learning*, Statistical Learning et Classification ; *Python Data Science Handbook*, Machine Learning.